# Au+Au minimum-bias direct-photon comparisons

This notebook reads the generator-corrected \(R_\gamma\) and direct-photon
spectrum exported by the analysis notebook. It contains only three final plots:

1. \(R_\gamma\) compared with published Au+Au centrality selections;
2. direct-photon spectrum compared with published Au+Au and the
   \(N_{\rm coll}\)-scaled p+p fit;
3. integrated direct-photon \(dN/dy\) versus charged-particle multiplicity.

The PHENIX logo and PDF output are disabled by default.


In [1]:
# ---------------- User configuration ----------------

FORMAT_NOTEBOOK = "../dca/input/Format.ipynb"

RGAMMA_CSV = "output/final_arrays/auau_mb_rgamma_generator_corrected.csv"
DIRECT_GAMMA_CSV = "output/final_arrays/auau_mb_direct_gamma.csv"

OUTPUT_DIRECTORY = "output/res/"

safe_to_pdf = False
draw_logo = False

# Au+Au 0-93% values used for comparison/labels.
NCOLL_AUAU_MB = 251.0

# Set this to the approved 0-93% Au+Au dNch/deta value and uncertainty.
# If the direct-spectrum CSV already contains dNch and dNch_err columns,
# those values are used instead.
DNCH_AUAU_MB = 185.0
DNCH_AUAU_MB_ERR = 12.0

# Integrated-yield interval.
DNDY_PT_MIN = 1.0
DNDY_PT_MAX = 5.0

# Statistical uncertainties are treated as uncorrelated between pT bins.
# Systematic uncertainties are treated as fully correlated by default.
STAT_BIN_CORRELATION = "uncorrelated"
SYST_BIN_CORRELATION = "fully_correlated"


In [2]:
CAP_UNCERTAINTIES = True
MAX_RELATIVE_UNCERTAINTY = 0.90

In [3]:
%run $FORMAT_NOTEBOOK

from pathlib import Path
from array import array

import numpy as np
import pandas as pd
import ROOT as root

root.TH1.AddDirectory(False)
root.gStyle.SetOptStat(0)
%jsroot on

Path(OUTPUT_DIRECTORY).mkdir(parents=True, exist_ok=True)


/home/yoren/.local/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Welcome to JupyROOT 6.30/06


Error in <TUnixSystem::FindDynamicLibrary>: input/logo/PHENIXTools/lib/libLogoPainter.so does not exist in /home/yoren/bnl/ROOT/install/lib:.:/home/yoren/bnl/ROOT/install/lib:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/lib/x86_64-linux-gnu/tls/haswell/x86_64:/lib/x86_64-linux-gnu/tls/haswell:/lib/x86_64-linux-gnu/tls/x86_64:/lib/x86_64-linux-gnu/tls:/lib/x86_64-linux-gnu/haswell/x86_64:/lib/x86_64-linux-gnu/haswell:/lib/x86_64-linux-gnu/x86_64:/lib/x86_64-linux-gnu:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/usr/lib/x86_64-linux-gnu/tls/haswell/x86_64:/usr/lib/x86_64-linux-gnu/tls/haswell:/usr/lib/x86_64-linux-gnu/tls/x86_64:/usr/lib/x86_64-linux-gnu/tls:/usr/lib/x86_64-linux-gnu/haswell/x86_64:/usr/lib/x86_64-linux-gnu/haswell:/usr/lib/x86_64-linux-gnu/x86_64:/usr/lib/x86_64-linux-gnu:/lib/glibc-hwcaps/x86-64-v3:/lib/glibc-hwcaps/x86-64-v2:/lib/tls/haswell/x86_64:/lib/tls/haswell:/lib

## Load and validate the extracted results


In [4]:
df_rgamma = pd.read_csv(RGAMMA_CSV)
df_direct = pd.read_csv(DIRECT_GAMMA_CSV)

required_rgamma = {
    "pt_low", "pt_high", "pt", "pt_err",
    "Rgamma",
    "Rgamma_stat_down", "Rgamma_stat_up",
    "Rgamma_syst_down", "Rgamma_syst_up",
}

required_direct = {
    "pt_low", "pt_high", "pt", "pt_err",
    "gamma_direct",
    "gamma_direct_stat",
    "gamma_direct_syst_down",
    "gamma_direct_syst_up",
}

missing_rgamma = required_rgamma - set(df_rgamma.columns)
missing_direct = required_direct - set(df_direct.columns)

if missing_rgamma:
    raise KeyError(f"Rgamma CSV is missing: {sorted(missing_rgamma)}")
if missing_direct:
    raise KeyError(f"Direct-photon CSV is missing: {sorted(missing_direct)}")

df_rgamma = df_rgamma.sort_values("pt").reset_index(drop=True)
df_direct = df_direct.sort_values("pt").reset_index(drop=True)

display(df_rgamma)
display(df_direct)


,pt_low,pt_high,pt,pt_err,r_ee_fit,r_ee_fit_stat,generator_extrapolation_factor,direct_to_decay,r,r_stat_down,r_stat_up,Rgamma,Rgamma_stat_down,Rgamma_stat_up,Rgamma_syst_down,Rgamma_syst_up,generated_cocktail_fit_integral,generated_direct_fit_integral,generated_cocktail_full_integral,generated_direct_full_integral
0,0.8,1.2,1.0,0.2,0.171341,0.020503,0.472829,0.097767,0.089060,0.011578,0.011856,1.097767,0.013777,0.014476,0.031437,3.532958e-02,0.004477,0.002076,0.026581,0.005827
1,1.2,1.6,1.4,0.2,0.181878,0.020312,0.489106,0.108734,0.098071,0.011938,0.012214,1.108734,0.014483,0.015221,0.029639,3.290076e-02,0.001085,0.000527,0.006256,0.001485
2,1.6,2.0,1.8,0.2,0.249212,0.023015,0.491107,0.163015,0.140166,0.014628,0.015026,1.163015,0.019455,0.020686,0.002541,2.560850e-03,0.000290,0.000145,0.001662,0.000409
3,2.0,2.8,2.4,0.4,0.243456,0.030620,0.493965,0.158958,0.137156,0.019333,0.020028,1.158958,0.025398,0.027541,0.005835,5.941085e-03,0.000116,0.000059,0.000660,0.000166
4,2.8,3.6,3.2,0.4,0.382623,0.052966,0.496000,0.307399,0.235122,0.039033,0.041703,1.307399,0.063480,0.075394,0.124960,1.813849e-01,0.000015,0.000008,0.000087,0.000022
5,3.6,4.2,3.9,0.3,0.371519,0.081099,0.499911,0.295517,0.228107,0.058254,0.064362,1.295517,0.090911,0.117850,0.295517,1.675832e+00,0.000003,0.000002,0.000017,0.000004
6,4.2,6.0,5.1,0.9,0.309045,0.081669,0.500620,0.223913,0.182949,0.054539,0.060066,1.223913,0.076585,0.097116,0.223913,5.006204e+08,0.000002,0.000001,0.000011,0.000003


,pt_low,pt_high,pt,pt_err,Rgamma,Rgamma_stat_down,Rgamma_stat_up,Rgamma_syst_down,Rgamma_syst_up,gamma_decay,gamma_direct,gamma_direct_stat,gamma_direct_syst_down,gamma_direct_syst_up
0,0.8,1.2,1.0,0.2,1.097767,0.013777,0.014476,0.031437,3.532958e-02,2.004821,0.196005,0.028322,0.063025,0.070829
1,1.2,1.6,1.4,0.2,1.108734,0.014483,0.015221,0.029639,3.290076e-02,0.336692,0.036610,0.005000,0.009979,0.011077
2,1.6,2.0,1.8,0.2,1.163015,0.019455,0.020686,0.002541,2.560850e-03,0.072009,0.011739,0.001445,0.000183,0.000184
3,2.0,2.8,2.4,0.4,1.158958,0.025398,0.027541,0.005835,5.941085e-03,0.011552,0.001836,0.000306,0.000067,0.000069
4,2.8,3.6,3.2,0.4,1.307399,0.063480,0.075394,0.124960,1.813849e-01,0.001209,0.000372,0.000084,0.000151,0.000219
5,3.6,4.2,3.9,0.3,1.295517,0.090911,0.117850,0.295517,1.675832e+00,0.000229,0.000068,0.000024,0.000068,0.000384
6,4.2,6.0,5.1,0.9,1.223913,0.076585,0.097116,0.223913,5.006204e+08,0.000033,0.000007,0.000003,0.000007,16720.839365


In [5]:
# ============================================================
# Optionally cap uncertainties at a fraction of central value
# ============================================================

CAP_UNCERTAINTIES = True
MAX_RELATIVE_UNCERTAINTY = 0.90


def cap_uncertainty_columns(
    dataframe,
    value_column,
    uncertainty_columns,
    relative_limit=0.90,
):
    """
    Cap each uncertainty at:

        uncertainty <= relative_limit * abs(central value)

    Original uncertainties are preserved in columns ending in '_original'.
    The operation is safe to rerun: the original columns are created only once.
    """
    if value_column not in dataframe.columns:
        raise KeyError(
            f"Central-value column '{value_column}' is missing."
        )

    central_value = np.abs(
        dataframe[value_column].to_numpy(dtype=float)
    )
    maximum_uncertainty = relative_limit * central_value

    for uncertainty_column in uncertainty_columns:
        if uncertainty_column not in dataframe.columns:
            print(
                f"Skipping missing uncertainty column: "
                f"{uncertainty_column}"
            )
            continue

        original_column = uncertainty_column + "_original"

        # Preserve the initial values the first time this cell is run.
        if original_column not in dataframe.columns:
            dataframe[original_column] = dataframe[
                uncertainty_column
            ].copy()

        original_uncertainty = np.abs(
            dataframe[original_column].to_numpy(dtype=float)
        )

        dataframe[uncertainty_column] = np.minimum(
            original_uncertainty,
            maximum_uncertainty,
        )

    return dataframe


if CAP_UNCERTAINTIES:
    # Cap Rgamma statistical and systematic uncertainties.
    df_rgamma = cap_uncertainty_columns(
        dataframe=df_rgamma,
        value_column="Rgamma",
        uncertainty_columns=[
            "Rgamma_stat_down",
            "Rgamma_stat_up",
            "Rgamma_syst_down",
            "Rgamma_syst_up",
        ],
        relative_limit=MAX_RELATIVE_UNCERTAINTY,
    )

    # Cap direct-photon spectrum uncertainties.
    df_direct = cap_uncertainty_columns(
        dataframe=df_direct,
        value_column="gamma_direct",
        uncertainty_columns=[
            "gamma_direct_stat",
            "gamma_direct_stat_down",
            "gamma_direct_stat_up",
            "gamma_direct_syst_down",
            "gamma_direct_syst_up",
        ],
        relative_limit=MAX_RELATIVE_UNCERTAINTY,
    )

    print(
        "Uncertainties capped at",
        f"{100 * MAX_RELATIVE_UNCERTAINTY:.0f}% of the central value.",
    )

else:
    print("Uncertainty capping is disabled.")

Skipping missing uncertainty column: gamma_direct_stat_down
Skipping missing uncertainty column: gamma_direct_stat_up
Uncertainties capped at 90% of the central value.


## Published Au+Au reference arrays used in the earlier Cu+Au comparison


In [6]:
# Rgamma reference points from the previous comparison notebook.

pt_ppg243 = np.array([
    0.9, 1.1, 1.3, 1.5, 1.7, 1.9,
    2.25, 2.75, 3.25, 3.75, 4.5, 6.0, 8.5,
])

RGAMMA_REFERENCE = {
    "30-40%": {
        "value": np.array([
            1.132, 1.140, 1.145, 1.151, 1.151, 1.195, 1.218,
            1.217, 1.250, 1.286, 1.346, 1.436, 1.832,
        ]),
        "stat": np.array([
            0.018, 0.014, 0.014, 0.015, 0.016, 0.019, 0.018,
            0.023, 0.035, 0.032, 0.052, 0.082, 0.247,
        ]),
        "syst": np.array([
            0.052, 0.052, 0.054, 0.053, 0.052, 0.054, 0.055,
            0.054, 0.055, 0.055, 0.055, 0.058, 0.074,
        ]),
    },
    "40-50%": {
        "value": np.array([
            1.111, 1.138, 1.147, 1.154, 1.132, 1.142, 1.171,
            1.223, 1.214, 1.310, 1.303, 1.398, 1.499,
        ]),
        "stat": np.array([
            0.017, 0.013, 0.013, 0.014, 0.015, 0.017, 0.015,
            0.023, 0.033, 0.054, 0.052, 0.080, 0.185,
        ]),
        "syst": np.array([
            0.052, 0.053, 0.054, 0.054, 0.053, 0.053, 0.054,
            0.055, 0.054, 0.057, 0.055, 0.057, 0.061,
        ]),
    },
    "50-60%": {
        "value": np.array([
            1.117, 1.145, 1.114, 1.144, 1.149, 1.138, 1.164,
            1.159, 1.203, 1.182, 1.176, 1.202, 1.816,
        ]),
        "stat": np.array([
            0.016, 0.019, 0.012, 0.013, 0.015, 0.017, 0.014,
            0.022, 0.033, 0.047, 0.051, 0.072, 0.297,
        ]),
        "syst": np.array([
            0.054, 0.055, 0.053, 0.055, 0.055, 0.054, 0.054,
            0.053, 0.054, 0.052, 0.053, 0.050, 0.074,
        ]),
    },
}


In [7]:
# Published Au+Au direct-photon invariant-yield reference used previously.

pt_auau_spectrum = np.array([
    0.9, 1.1, 1.3, 1.5, 1.7, 1.9,
    2.25, 2.75, 3.25, 3.75, 4.5, 6.0, 8.5,
])

auau_40_50_yield = np.array([
    1.785e-01, 8.562e-02, 3.786e-02, 1.798e-02,
    7.225e-03, 3.809e-03, 1.445e-03, 4.402e-04,
    1.170e-04, 5.412e-05, 1.214e-05, 1.503e-06,
    1.037e-07,
])

auau_40_50_stat = np.array([
    2.674e-02, 8.298e-03, 3.393e-03, 1.642e-03,
    8.086e-04, 4.485e-04, 1.263e-04, 4.555e-05,
    1.830e-05, 9.488e-06, 2.098e-06, 3.014e-07,
    3.870e-08,
])

auau_40_50_syst = np.array([
    8.115e-02, 3.212e-02, 1.345e-02, 6.122e-03,
    2.781e-03, 1.369e-03, 4.507e-04, 1.142e-04,
    3.097e-05, 1.191e-05, 2.718e-06, 3.318e-07,
    2.463e-08,
])

# p+p direct-photon fit used in the previous notebook.
PP_DIRECT_FIT = {
    "A": 1.60e-4,
    "p0": 1.45,
    "n": 3.3,
}

def ncoll_scaled_pp_reference(pt):
    pt = np.asarray(pt, float)
    p = PP_DIRECT_FIT
    return (
        NCOLL_AUAU_MB
        * p["A"]
        / (1.0 + (pt / p["p0"])**2)**p["n"]
    )


In [8]:
# Published Au+Au direct-photon invariant-yield reference:
# 20-40% Au+Au, sqrt(sNN) = 200 GeV

pt_auau_spectrum = np.array([
    0.9, 1.1, 1.3, 1.5, 1.7, 1.9,
    2.25, 2.75, 3.25, 3.75, 4.5, 6.0, 8.5,
])

auau_20_40_yield = np.array([
    3.983e-01,
    1.722e-01,
    7.048e-02,
    3.187e-02,
    1.595e-02,
    9.016e-03,
    3.023e-03,
    5.774e-04,
    1.835e-04,
    5.850e-05,
    7.887e-06,
    7.163e-07,
   -1.058e-07,
])

auau_20_40_stat_down = np.array([
    4.848e-02,
    1.461e-02,
    5.797e-03,
    2.671e-03,
    1.596e-03,
    7.562e-04,
    2.132e-04,
    6.770e-05,
    2.861e-05,
    1.343e-05,
    2.951e-06,
    4.846e-07,
    0.000e+00,
])

auau_20_40_stat_up = np.array([
    4.848e-02,
    1.461e-02,
    5.797e-03,
    2.671e-03,
    1.596e-03,
    7.562e-04,
    2.132e-04,
    6.770e-05,
    2.861e-05,
    1.343e-05,
    2.951e-06,
    4.846e-07,
    1.543e-07,
])

auau_20_40_syst_down = np.array([
    1.683e-01,
    6.606e-02,
    2.765e-02,
    1.234e-02,
    5.834e-03,
    2.930e-03,
    9.805e-04,
    2.331e-04,
    7.517e-05,
    2.735e-05,
    7.102e-06,
    7.163e-07,
    0.000e+00,
])

auau_20_40_syst_up = np.array([
    1.683e-01,
    6.606e-02,
    2.765e-02,
    1.234e-02,
    5.834e-03,
    2.930e-03,
    9.805e-04,
    2.331e-04,
    7.517e-05,
    2.735e-05,
    7.102e-06,
    1.434e-06,
    2.001e-07,
])

In [15]:
pt_auau_spectrum = np.array([0.9, 1.1, 1.3, 1.5, 1.7, 1.9, 2.25, 2.75, 3.25, 3.75, 4.5, 6.0])
pt_auau_30_40_err = np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.25, 0.25, 0.25, 0.25, 0.5, 1.0])

auau_30_40_yield = np.array([3.573e-1, 1.457e-1, 6.346e-2, 2.928e-2, 1.363e-2, 8.574e-3, 3.011e-3, 6.748e-4, 2.120e-4, 7.667e-5, 2.077e-5, 2.396e-6])

auau_30_40_stat = np.array([4.885e-2, 1.485e-2, 6.090e-3, 2.845e-3, 1.414e-3, 8.174e-4, 2.282e-4, 7.177e-5, 2.982e-5, 1.386e-5, 2.162e-6, 4.530e-7])

auau_30_40_syst = np.array([1.361e-1, 5.301e-2, 2.226e-2, 9.913e-3, 4.582e-3, 2.347e-3, 7.772e-4, 1.170e-4, 5.119e-5, 1.727e-5, 4.492e-6, np.nan])

In [16]:
# Published Au+Au integrated direct-photon yields, 1 < pT < 5 GeV/c.

dNch_auau = np.array([
    623.9, 414.2, 274.0, 176.8,
    109.4, 61.6, 32.0, 15.5,
])

dNch_auau_err = 0.05 * dNch_auau

dndy_auau = np.array([
    1.250e+00, 9.956e-01, 7.202e-01, 4.549e-01,
    2.626e-01, 1.358e-01, 5.746e-02, 2.700e-02,
])

dndy_auau_stat = np.array([
    1.594e-01, 8.483e-02, 3.902e-02, 2.554e-02,
    1.433e-02, 7.254e-03, 3.696e-03, 1.880e-03,
])

dndy_auau_syst = np.array([
    5.212e-01, 3.510e-01, 2.360e-01, 1.556e-01,
    9.418e-02, 5.193e-02, 2.455e-02, 1.054e-02,
])


## \(R_\gamma\) comparison


In [17]:
_rgamma_keepalive = {}

canvas = root.TCanvas("c_rgamma_AuAu_MB_comparison", "Rgamma", 850, 650)
canvas.SetTicks(1, 1)

x_min = min(0.7, float(df_rgamma["pt_low"].min()))
x_max = max(7.0, float(df_rgamma["pt_high"].max()))

frame = root.TH1D(
    "frame_rgamma_AuAu_MB_comparison",
    ";p_{T} [GeV/c];R_{#gamma}",
    100, x_min, x_max,
)
frame.SetDirectory(0)
frame.SetStats(0)
frame.SetMinimum(0.91)

reference_max = max(
    np.max(values["value"] + values["syst"])
    for values in RGAMMA_REFERENCE.values()
)
data_max = np.max(
    df_rgamma["Rgamma"] + df_rgamma["Rgamma_syst_up"]
)
frame.SetMaximum(min(1.85, 1.10 * reference_max, 1.10 * data_max))
frame.Draw()

reference_styles = {
    "30-40%": (22, root.kMagenta),
    "40-50%": (20, root.kRed),
    "50-60%": (23, root.kGreen + 2),
}

reference_graphs = {}

for label, values in RGAMMA_REFERENCE.items():
    marker, color = reference_styles[label]

    g_stat = root.TGraphErrors(len(pt_ppg243))
    g_syst = root.TGraphErrors(len(pt_ppg243))

    for i, pt in enumerate(pt_ppg243):
        g_stat.SetPoint(i, float(pt), float(values["value"][i]))
        g_stat.SetPointError(i, 0.0, float(values["stat"][i]))

        g_syst.SetPoint(i, float(pt), float(values["value"][i]))
        g_syst.SetPointError(i, 0.1, float(values["syst"][i]))

    g_syst.SetFillColorAlpha(color, 0.35)
    g_syst.SetLineColor(color)
    g_stat.SetMarkerStyle(marker)
    g_stat.SetMarkerSize(1.3)
    g_stat.SetMarkerColor(color)
    g_stat.SetLineColor(color)

    g_syst.Draw("2 SAME")
    g_stat.Draw("P SAME")
    reference_graphs[label] = (g_stat, g_syst)

g_data_syst = root.TGraphAsymmErrors(len(df_rgamma))
g_data_stat = root.TGraphAsymmErrors(len(df_rgamma))

for i, row in enumerate(df_rgamma.itertuples(index=False)):
    g_data_syst.SetPoint(i, row.pt, row.Rgamma)
    g_data_syst.SetPointError(
        i, row.pt_err, row.pt_err,
        row.Rgamma_syst_down, row.Rgamma_syst_up,
    )

    g_data_stat.SetPoint(i, row.pt, row.Rgamma)
    g_data_stat.SetPointError(
        i, 0.0, 0.0,
        row.Rgamma_stat_down, row.Rgamma_stat_up,
    )

g_data_syst.SetFillColorAlpha(root.kBlack, 0.30)
g_data_syst.SetLineColor(root.kBlack)
g_data_stat.SetMarkerStyle(33)
g_data_stat.SetMarkerSize(2.0)
g_data_stat.SetMarkerColor(root.kBlack)
g_data_stat.SetLineColor(root.kBlack)

g_data_syst.Draw("2 SAME")
g_data_stat.Draw("P SAME")

unity = root.TLine(x_min, 1.0, x_max, 1.0)
unity.SetLineStyle(2)
unity.Draw("SAME")

legend = root.TLegend(0.16, 0.57, 0.55, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.SetTextSize(0.041)
legend.AddEntry(g_data_stat, "0-93% Au+Au, this analysis", "p")
for label in ("30-40%", "40-50%", "50-60%"):
    legend.AddEntry(reference_graphs[label][0], f"{label} Au+Au, PPG243", "p")
legend.Draw()

title = root.TLatex()
title.SetNDC(True)
title.SetTextSize(0.047)
title.DrawLatex(0.16, 0.92, "Au+Au, #sqrt{s_{NN}} = 200 GeV, |#eta|<0.35")

if draw_logo:
    root.PHENIXTools.DrawPreliminary(0.70, 0.17, 0.23)

canvas.Draw()

if safe_to_pdf:
    canvas.SaveAs(OUTPUT_DIRECTORY + "auau_mb_rgamma_comparison_logo.pdf")

_rgamma_keepalive.update({
    "canvas": canvas,
    "frame": frame,
    "data_stat": g_data_stat,
    "data_syst": g_data_syst,
    "references": reference_graphs,
    "unity": unity,
    "legend": legend,
    "title": title,
})


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_rgamma_AuAu_MB_comparison


## Direct-photon spectrum comparison


In [29]:
_spectrum_keepalive = {}

valid = (
    np.isfinite(df_direct["gamma_direct"])
    & (df_direct["gamma_direct"] > 0)
)
plot_data = df_direct.loc[valid].copy()

if plot_data.empty:
    raise RuntimeError("No positive direct-photon points are available.")

canvas = root.TCanvas(
    "c_direct_AuAu_MB_comparison",
    "Direct-photon spectrum",
    850,
    650,
)
canvas.SetTicks(1, 1)
canvas.SetLogy()

x_min = min(0.7, float(plot_data["pt_low"].min()))
x_max = max(7.0, float(plot_data["pt_high"].max()))

pt_curve = np.linspace(max(0.7, x_min), x_max, 400)
pp_curve = ncoll_scaled_pp_reference(pt_curve)

positive_min = min(
    float(plot_data["gamma_direct"].min()),
    float(auau_30_40_yield.min()),
    float(pp_curve.min()),
)
positive_max = max(
    float(
        (
            plot_data["gamma_direct"]
            + plot_data["gamma_direct_syst_up"]
        ).max()
    ),
    float((auau_30_40_yield + auau_30_40_syst).max()),
    float(pp_curve.max()),
)

frame = root.TH1D(
    "frame_direct_AuAu_MB_comparison",
    (
        ";p_{T} [GeV/c];"
        "d^{2}N_{#gamma}^{dir}/(2#pip_{T}dp_{T}d#eta) "
        "[(GeV/c)^{-2}]"
    ),
    100, x_min, x_max,
)
frame.SetDirectory(0)
frame.SetStats(0)
frame.SetMinimum(max(1e-12, 0.20 * positive_min))
frame.SetMaximum(5.0 * positive_max)
frame.Draw()

g_ref_syst = root.TGraphErrors(len(pt_auau_spectrum))
g_ref_stat = root.TGraphErrors(len(pt_auau_spectrum))

for i, pt in enumerate(pt_auau_spectrum):
    g_ref_syst.SetPoint(i, float(pt), float(auau_30_40_yield[i]))
    g_ref_syst.SetPointError(i, 0.05, float(auau_30_40_syst[i]))

    g_ref_stat.SetPoint(i, float(pt), float(auau_30_40_yield[i]))
    g_ref_stat.SetPointError(i, 0.0, float(auau_30_40_stat[i]))

g_ref_syst.SetFillColorAlpha(root.kRed, 0.35)
g_ref_syst.SetLineColor(root.kRed)
g_ref_stat.SetMarkerStyle(20)
g_ref_stat.SetMarkerSize(1.2)
g_ref_stat.SetMarkerColor(root.kRed)
g_ref_stat.SetLineColor(root.kRed)

g_pp = root.TGraph(len(pt_curve))
for i, (pt, value) in enumerate(zip(pt_curve, pp_curve)):
    g_pp.SetPoint(i, float(pt), float(value))
g_pp.SetLineColor(root.kOrange + 7)
g_pp.SetLineWidth(3)

g_data_syst = root.TGraphAsymmErrors(len(plot_data))
g_data_stat = root.TGraphErrors(len(plot_data))

for i, row in enumerate(plot_data.itertuples(index=False)):
    g_data_syst.SetPoint(i, row.pt, row.gamma_direct)
    g_data_syst.SetPointError(
        i, row.pt_err, row.pt_err,
        row.gamma_direct_syst_down,
        row.gamma_direct_syst_up,
    )

    g_data_stat.SetPoint(i, row.pt, row.gamma_direct)
    g_data_stat.SetPointError(i, 0.0, row.gamma_direct_stat)

g_data_syst.SetFillColorAlpha(root.kBlack, 0.30)
g_data_syst.SetLineColor(root.kBlack)
g_data_stat.SetMarkerStyle(33)
g_data_stat.SetMarkerSize(2.0)
g_data_stat.SetMarkerColor(root.kBlack)
g_data_stat.SetLineColor(root.kBlack)

g_ref_syst.Draw("2 SAME")
g_ref_stat.Draw("P SAME")
g_pp.Draw("L SAME")
g_data_syst.Draw("2 SAME")
g_data_stat.Draw("P SAME")

legend = root.TLegend(0.16, 0.16, 0.63, 0.37)
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.SetTextSize(0.040)
legend.AddEntry(g_data_stat, "0-93% Au+Au, this analysis", "p")
legend.AddEntry(g_ref_stat, "30-40% Au+Au, PPG243", "p")
legend.AddEntry(
    g_pp,
    f"#LTN_{{coll}}#GT={NCOLL_AUAU_MB:.0f} scaled p+p fit",
    "l",
)
legend.Draw()

title = root.TLatex()
title.SetNDC(True)
title.SetTextSize(0.047)
title.DrawLatex(0.16, 0.92, "Au+Au, #sqrt{s_{NN}} = 200 GeV, |#eta|<0.35")

if draw_logo:
    root.PHENIXTools.DrawPreliminary(0.70, 0.50, 0.23)

canvas.Draw()

if safe_to_pdf:
    canvas.SaveAs(OUTPUT_DIRECTORY + "auau_mb_direct_spectrum_logo.pdf")

_spectrum_keepalive.update({
    "canvas": canvas,
    "frame": frame,
    "data_stat": g_data_stat,
    "data_syst": g_data_syst,
    "reference_stat": g_ref_stat,
    "reference_syst": g_ref_syst,
    "pp_reference": g_pp,
    "legend": legend,
    "title": title,
})


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_direct_AuAu_MB_comparison


## Integrate the extracted spectrum


In [19]:
def integrate_direct_photon_yield(
    dataframe,
    pt_min=DNDY_PT_MIN,
    pt_max=DNDY_PT_MAX,
):
    """
    Integrate invariant yield:
        dN/dy = integral 2*pi*pT * invariant_yield dpT.

    Statistical errors are combined in quadrature when uncorrelated.
    Systematic errors are summed linearly when fully correlated.
    """
    value = 0.0
    stat_terms = []
    syst_down_terms = []
    syst_up_terms = []
    rows = []

    for row in dataframe.itertuples(index=False):
        if not np.isfinite(row.gamma_direct):
            continue

        low = max(float(row.pt_low), float(pt_min))
        high = min(float(row.pt_high), float(pt_max))
        if high <= low:
            continue

        width = high - low
        weight = 2.0 * np.pi * float(row.pt) * width

        contribution = weight * float(row.gamma_direct)
        stat = weight * float(row.gamma_direct_stat)
        syst_down = weight * float(row.gamma_direct_syst_down)
        syst_up = weight * float(row.gamma_direct_syst_up)

        value += contribution
        stat_terms.append(stat)
        syst_down_terms.append(syst_down)
        syst_up_terms.append(syst_up)

        rows.append({
            "pt": row.pt,
            "pt_low_used": low,
            "pt_high_used": high,
            "contribution": contribution,
            "stat_contribution": stat,
            "syst_down_contribution": syst_down,
            "syst_up_contribution": syst_up,
        })

    if STAT_BIN_CORRELATION == "uncorrelated":
        stat_total = float(np.sqrt(np.sum(np.square(stat_terms))))
    elif STAT_BIN_CORRELATION == "fully_correlated":
        stat_total = float(np.sum(np.abs(stat_terms)))
    else:
        raise ValueError("Unknown STAT_BIN_CORRELATION")

    if SYST_BIN_CORRELATION == "fully_correlated":
        syst_down_total = float(np.sum(np.abs(syst_down_terms)))
        syst_up_total = float(np.sum(np.abs(syst_up_terms)))
    elif SYST_BIN_CORRELATION == "uncorrelated":
        syst_down_total = float(np.sqrt(np.sum(np.square(syst_down_terms))))
        syst_up_total = float(np.sqrt(np.sum(np.square(syst_up_terms))))
    else:
        raise ValueError("Unknown SYST_BIN_CORRELATION")

    return {
        "dNdy": value,
        "stat": stat_total,
        "syst_down": syst_down_total,
        "syst_up": syst_up_total,
        "rows_used": pd.DataFrame(rows),
    }


integrated = integrate_direct_photon_yield(df_direct)
display(integrated["rows_used"])

print(
    f"dN/dy, {DNDY_PT_MIN:g} < pT < {DNDY_PT_MAX:g} GeV/c = "
    f"{integrated['dNdy']:.6g} "
    f"+/- {integrated['stat']:.3g} (stat) "
    f"- {integrated['syst_down']:.3g} "
    f"+ {integrated['syst_up']:.3g} (syst)"
)


,pt,pt_low_used,pt_high_used,contribution,stat_contribution,syst_down_contribution,syst_up_contribution
0,1.0,1.0,1.2,0.246307,0.035590,0.079199,0.089007
1,1.4,1.2,1.6,0.128815,0.017595,0.035112,0.038977
2,1.8,1.6,2.0,0.053104,0.006538,0.000828,0.000834
3,2.4,2.0,2.8,0.022153,0.003689,0.000813,0.000828
4,3.2,2.8,3.6,0.005976,0.001350,0.002429,0.003526
5,3.9,3.6,4.2,0.000995,0.000351,0.000895,0.000895
6,5.1,4.2,5.0,0.000192,0.000074,0.000173,0.000173


dN/dy, 1 < pT < 5 GeV/c = 0.457542 +/- 0.0404 (stat) - 0.119 + 0.134 (syst)


In [20]:
# Resolve the Au+Au MB multiplicity from the CSV or configuration.

if "dNch" in df_direct.columns and np.isfinite(df_direct["dNch"].iloc[0]):
    dNch_mb = float(df_direct["dNch"].iloc[0])
else:
    if DNCH_AUAU_MB is None:
        raise RuntimeError(
            "Set DNCH_AUAU_MB in the configuration cell, or add a dNch "
            "column to the direct-photon CSV."
        )
    dNch_mb = float(DNCH_AUAU_MB)

if "dNch_err" in df_direct.columns and np.isfinite(df_direct["dNch_err"].iloc[0]):
    dNch_mb_err = float(df_direct["dNch_err"].iloc[0])
else:
    if DNCH_AUAU_MB_ERR is None:
        raise RuntimeError(
            "Set DNCH_AUAU_MB_ERR in the configuration cell, or add a "
            "dNch_err column to the direct-photon CSV."
        )
    dNch_mb_err = float(DNCH_AUAU_MB_ERR)

df_integrated = pd.DataFrame([{
    "system": "AuAu",
    "centrality": "0-93%",
    "pt_min": DNDY_PT_MIN,
    "pt_max": DNDY_PT_MAX,
    "dNch": dNch_mb,
    "dNch_err": dNch_mb_err,
    "dNdy": integrated["dNdy"],
    "dNdy_stat": integrated["stat"],
    "dNdy_syst_down": integrated["syst_down"],
    "dNdy_syst_up": integrated["syst_up"],
}])

display(df_integrated)

if safe_to_pdf:
    integrated_csv = (
        "../dca/output/final_integrated/"
        "auau_mb_direct_gamma_dndy_1_5.csv"
    )
    Path(integrated_csv).parent.mkdir(parents=True, exist_ok=True)
    df_integrated.to_csv(integrated_csv, index=False)
    print("Saved:", integrated_csv)


,system,centrality,pt_min,pt_max,dNch,dNch_err,dNdy,dNdy_stat,dNdy_syst_down,dNdy_syst_up
0,AuAu,0-93%,1.0,5.0,185.0,12.0,0.457542,0.040429,0.11945,0.13424


## Integrated-yield comparison


In [21]:
# ============================================================
# Integrated direct-photon yield versus dNch/deta
# Styled like the original int_spectrum.ipynb final plot
# ============================================================

pt_min = DNDY_PT_MIN
pt_max = DNDY_PT_MAX
pt_label = f"{pt_min:g}<p_{{T}}<{pt_max:g} GeV/c"

_dndy_keepalive = {}

c_dndy_vs_dNch = root.TCanvas(
    "c_dndy_vs_dNch_AuAu_MB",
    "Integrated direct photons",
    850,
    700,
)
c_dndy_vs_dNch.SetLogy()
c_dndy_vs_dNch.SetLogx()
c_dndy_vs_dNch.SetTicks(1, 1)

frame = root.TH2F(
    "frame_dndy_vs_dNch_AuAu_MB",
    f";dN_{{ch}}/d#eta;dN_{{#gamma}}^{{dir}}/dy, {pt_label}",
    100,
    9.0,
    2.0e3,
    100,
    1.0e-4,
    20.0,
)
frame.SetDirectory(0)
frame.SetStats(0)

Format_Hist_total(
    frame,
    title_x="dN_{ch}/d#eta",
    title_y="dN_{#gamma}^{dir}/dy",
    left=0.15,
    bottom=0.15,
    right=0.01,
    top=0.01,
    Tsize=0.06,
    Lsize=0.06,
    Mstyle=21,
    Msize=1.2,
    Mcolor=4,
    Lwidth=3,
    Lcolor=4,
    offset_x=1.2,
    offset_y=1.2,
    title="",
    Malpha=0.5,
    Lalpha=1,
)

frame.Draw()


# ------------------------------------------------------------
# Published Au+Au points
# ------------------------------------------------------------

g_auau_ppg_syst = root.TGraphErrors(len(dNch_auau))
g_auau_ppg_stat = root.TGraphErrors(len(dNch_auau))

for i in range(len(dNch_auau)):
    g_auau_ppg_syst.SetPoint(
        i,
        float(dNch_auau[i]),
        float(dndy_auau[i]),
    )
    g_auau_ppg_syst.SetPointError(
        i,
        float(dNch_auau_err[i]),
        float(dndy_auau_syst[i]),
    )

    g_auau_ppg_stat.SetPoint(
        i,
        float(dNch_auau[i]),
        float(dndy_auau[i]),
    )
    g_auau_ppg_stat.SetPointError(
        i,
        0.0,
        float(dndy_auau_stat[i]),
    )

g_auau_ppg_syst.SetFillColorAlpha(root.kRed, 0.22)
g_auau_ppg_syst.SetLineColor(root.kRed)

g_auau_ppg_stat.SetMarkerStyle(25)
g_auau_ppg_stat.SetMarkerSize(1.2)
g_auau_ppg_stat.SetMarkerColor(root.kRed)
g_auau_ppg_stat.SetLineColor(root.kRed)

# Draw systematic boxes before markers.
g_auau_ppg_syst.Draw("2 SAME")
g_auau_ppg_stat.Draw("P SAME")


# ------------------------------------------------------------
# This 0-93% Au+Au result
# ------------------------------------------------------------

g_mb_syst = root.TGraphAsymmErrors(1)
g_mb_syst.SetPoint(
    0,
    float(dNch_mb),
    float(integrated["dNdy"]),
)
g_mb_syst.SetPointError(
    0,
    float(dNch_mb_err),
    float(dNch_mb_err),
    float(integrated["syst_down"]),
    float(integrated["syst_up"]),
)

g_mb_syst.SetFillColorAlpha(root.kBlack, 0.40)
g_mb_syst.SetLineColor(root.kBlack)
g_mb_syst.Draw("2 SAME")

g_mb_stat = root.TGraphErrors(1)
g_mb_stat.SetPoint(
    0,
    float(dNch_mb),
    float(integrated["dNdy"]),
)
g_mb_stat.SetPointError(
    0,
    0.0,
    float(integrated["stat"]),
)

g_mb_stat.SetMarkerStyle(33)
g_mb_stat.SetMarkerSize(3.0)
g_mb_stat.SetMarkerColor(root.kBlack)
g_mb_stat.SetLineColor(root.kBlack)
g_mb_stat.Draw("P SAME")


# ------------------------------------------------------------
# Fit the published Au+Au points with A * (dNch/deta)^alpha
# ------------------------------------------------------------

f_scale_auau = root.TF1(
    "f_scale_auau_mb_comparison",
    "[0]*pow(x,[1])",
    10.0,
    1500.0,
)
f_scale_auau.SetParameters(1.0e-4, 1.25)
f_scale_auau.SetLineColor(root.kBlack)
f_scale_auau.SetLineStyle(6)
f_scale_auau.SetLineWidth(2)

# Use statistical uncertainties in the fit, matching the old notebook.
g_auau_ppg_stat.Fit(f_scale_auau, "RQ0")
f_scale_auau.Draw("SAME")


# ------------------------------------------------------------
# Main reference legend
# ------------------------------------------------------------

leg = root.TLegend(0.34, 0.18, 0.80, 0.49)
leg.SetBorderSize(0)
leg.SetFillStyle(0)
leg.SetTextSize(0.050)

leg.AddEntry(
    g_auau_ppg_stat,
    "PHENIX Au+Au 200 GeV 2025",
    "p",
)

# Optional archive/reference systems, when ref_graphs exists.
if "ref_graphs" in globals():
    added = set()

    for item in ref_graphs:
        name = item["group"]["name"]

        if name in added:
            continue

        leg.AddEntry(item["stat"], name, "p")
        added.add(name)

leg.Draw()


# ------------------------------------------------------------
# Analysis label and fitted scaling exponent
# ------------------------------------------------------------

leg0 = root.TLegend(
    0.18,
    0.74,
    0.60,
    0.96,
    f"     Direct #gamma ({pt_label})",
)
leg0.SetBorderSize(0)
leg0.SetFillStyle(0)
leg0.SetTextSize(0.050)

leg0.AddEntry(
    g_mb_stat,
    "0%-93% Au+Au 200 GeV",
    "p",
)

leg0.AddEntry(
    f_scale_auau,
    (
        f"#alpha^{{2025}}_{{Au+Au}} = "
        f"{f_scale_auau.GetParameter(1):.2f}"
    ),
    "l",
)

leg0.Draw()


# ------------------------------------------------------------
# Publication reference
# ------------------------------------------------------------

leg_reference = root.TLegend(
    0.60,
    0.50,
    0.82,
    0.56,
    "PRC109 044912",
)
leg_reference.SetBorderSize(0)
leg_reference.SetFillStyle(0)
leg_reference.SetTextSize(0.040)
leg_reference.Draw()


# ------------------------------------------------------------
# PHENIX label and optional output
# ------------------------------------------------------------

if draw_logo:
    root.PHENIXTools.DrawPreliminary(
        0.70,
        0.57,
        0.23,
    )

#c_dndy_vs_dNch.Modified()
#c_dndy_vs_dNch.Update()
c_dndy_vs_dNch.Draw()

if safe_to_pdf:
    c_dndy_vs_dNch.SaveAs(
        OUTPUT_DIRECTORY
        + "auau_mb_dndy_vs_dNch_logo.pdf"
    )


# Explicit PyROOT ownership.
_dndy_keepalive.update({
    "canvas": c_dndy_vs_dNch,
    "frame": frame,
    "auau_stat": g_auau_ppg_stat,
    "auau_syst": g_auau_ppg_syst,
    "mb_stat": g_mb_stat,
    "mb_syst": g_mb_syst,
    "scale_fit": f_scale_auau,
    "legend": leg,
    "analysis_legend": leg0,
    "reference_legend": leg_reference,
})

In [22]:
Nch_ref = np.array([
    341.175, 151.75, 131.525,
    104.264,
    109.263, 51.654,
    519.025, 225.425, 85.475, 16.362,
    519.025, 225.425, 186.11,
    1206.75
])

eNch_ref = np.array([
    29.325, 12.69, 11.152,
    8.882,
    7.813, 3.559,
    26.25, 13.175, 8.075, 2.8137,
    26.25, 13.175, 11.324,
    45.75
])

Yield_ref = np.array([
    1.28873, 0.415701, 0.336163,
    0.259703,
    0.152438, 0.0608154,
    1.66115, 0.707725, 0.217113, 0.0229508,
    1.68592, 0.698072, 0.554999,
    4.75833
])

eYield_ref = np.array([
    0.271747, 0.115494, 0.0652705,
    0.110752,
    0.0597752, 0.0212248,
    0.202514, 0.0863305, 0.0329216, 0.00620148,
    0.231152, 0.0854467, 0.050432,
    0.316776
])

sYield_ref = np.array([
    0.452504, 0.210051, 0.172416,
    0.2140941,
    0.0691723, 0.0252199,
    0.5339347, 0.2478384, 0.0936435, 0.0154706,
    0.37753, 0.149428, 0.117674,
    1.87478
])

In [23]:
# Group indices for Archive / ScalingData.h, 1 < pT < 5 GeV/c
groups_1_5 = [
    {
        "name": "PHENIX Au+Au 200 GeV 2015",
        "idx": [6, 7, 8, 9],
        "color": root.kRed,
        "marker": 21,   # filled square
        "filled": True,
    },
    {
        "name": "PHENIX Au+Au 200 GeV 2015",
        "idx": [10, 11, 12],
        "color": root.kRed,
        "marker": 25,   # open square
        "filled": False,
    },
    {
        "name": "PHENIX Au+Au 62.4 GeV",
        "idx": [0, 1, 2],
        "color": root.kBlue,
        "marker": 20,   # filled circle
        "filled": True,
    },
    {
        "name": "PHENIX Au+Au 39 GeV",
        "idx": [3],
        "color": root.kOrange + 7,
        "marker": 24,   # open circle
        "filled": False,
    },
    {
        "name": "PHENIX Cu+Cu 200 GeV",
        "idx": [4, 5],
        "color": root.kGreen + 2,
        "marker": 22,   # filled triangle
        "filled": True,
    },
    {
        "name": "ALICE Pb+Pb 2760 GeV",
        "idx": [13],
        "color": root.kOrange + 1,
        "marker": 33,   # diamond/star-like
        "filled": False,
    },
]

In [24]:
# Extra PHENIX Au+Au 200 GeV PPG / published points
# Direct gamma dN/dy, 1 < pT < 5 GeV/c

dNch_auau_ppg = np.array([
    623.9, 414.2, 274.0, 176.8,
    109.4, 61.6, 32.0, 15.5
])

# If you do not have dNch systematic/errors for these,
# use 5% as in the archive plotting macro.
dNch_auau_ppg_err = 0.05 * dNch_auau_ppg

y_auau_ppg = np.array([
    1.250e+00, 9.956e-01, 7.202e-01, 4.549e-01,
    2.626e-01, 1.358e-01, 5.746e-02, 2.700e-02
])

y_auau_ppg_stat = np.array([
    1.594e-01, 8.483e-02, 3.902e-02, 2.554e-02,
    1.433e-02, 7.254e-03, 3.696e-03, 1.880e-03
])

y_auau_ppg_syst = np.array([
    5.212e-01, 3.510e-01, 2.360e-01, 1.556e-01,
    9.418e-02, 5.193e-02, 2.455e-02, 1.054e-02
])

In [25]:
def make_group_graphs(Nch, eNch, Y, eY, sY, group, name):
    idx = group["idx"]
    n = len(idx)

    g_stat = root.TGraphErrors(n)
    g_syst = root.TGraphErrors(n)

    for ip, i in enumerate(idx):
        x = float(Nch[i])
        y = float(Y[i])

        # stat: y stat only
        g_stat.SetPoint(ip, x, y)
        g_stat.SetPointError(ip, 0.0, float(eY[i]))

        # syst: x syst + y syst box
        ex_syst = max(0.05 * x, float(eNch[i]))
        g_syst.SetPoint(ip, x, y)
        g_syst.SetPointError(ip, ex_syst, float(sY[i]))

    g_stat.SetName(f"g_stat_{name}")
    g_syst.SetName(f"g_syst_{name}")

    color = group["color"]
    marker = group["marker"]

    g_stat.SetMarkerStyle(marker)
    g_stat.SetMarkerSize(1.15)
    g_stat.SetMarkerColor(color)
    g_stat.SetLineColor(color)

    # open marker style already controls open/filled for ROOT marker codes
    if not group["filled"]:
        g_stat.SetMarkerColor(color)
        g_stat.SetLineColor(color)

    g_syst.SetFillColorAlpha(color, 0.25)
    g_syst.SetLineColor(color)

    return g_stat, g_syst

In [26]:
groups_to_plot = groups_1_5
#groups_to_plot = groups_15_5

In [27]:
ref_graphs = []

for ig, group in enumerate(groups_to_plot):
    g_stat, g_syst = make_group_graphs(
        Nch_ref,
        eNch_ref,
        Yield_ref,
        eYield_ref,
        sYield_ref,
        group,
        f"group{ig}"
    )

    ref_graphs.append({
        "group": group,
        "stat": g_stat,
        "syst": g_syst,
    })

In [28]:
pt_label = f"{pt_min:g}<p_{{T}}<{pt_max:g} GeV/c"

c_dndy_vs_dNch = root.TCanvas("c_dndy_vs_dNch", "c_dndy_vs_dNch", 850, 700)
c_dndy_vs_dNch.SetLogy()
c_dndy_vs_dNch.SetLogx()

frame = root.TH2F(
    "frame_dndy_vs_dNch",
    f";dN_{{ch}}/d#eta;dN_{{#gamma}}^{{dir}}/dy, {pt_label}",
    100, 9.0, 2e3,
    100, 1e-4, 20.0
)
frame.SetStats(0)

Format_Hist_total(frame, title_x="dN_{ch}/d#eta",  title_y="dN_{#gamma}^{dir}/dy", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.06,  Lsize=0.06,\
                      Mstyle=21,  Msize=1.2, Mcolor=4,  Lwidth=3,  Lcolor=4,  offset_x=1.2, offset_y=1.2, title="",  Malpha=0.5,  Lalpha=1)
frame.Draw()

# Draw systematic boxes first
for item in ref_graphs:
    item["syst"].Draw("2 same")

# Draw stat markers second
for item in ref_graphs:
    item["stat"].Draw("P same")

#new auau 
# Extra PPG Au+Au 200 systematic boxes
g_auau_ppg_syst.Draw("2 same")
g_auau_ppg_stat.Draw("P same")

# Your MB syst
g_mb_syst.SetFillColorAlpha(root.kBlack, 0.40)
g_mb_syst.SetLineColor(root.kBlack)
g_mb_syst.Draw("2 same")

# Your MB stat
g_mb_stat.SetMarkerStyle(33)
g_mb_stat.SetMarkerSize(3.0)
g_mb_stat.SetMarkerColor(root.kBlack)
g_mb_stat.SetLineColor(root.kBlack)
g_mb_stat.Draw("P same")

# Legend similar to published plot
leg = root.TLegend(0.34, 0.18, 0.8, 0.5)
leg.SetBorderSize(0)
leg.SetFillStyle(0)
leg.SetTextSize(0.050)

# Avoid duplicate legend entries for Au+Au 200 if filled/open are both present
added = set()

leg.AddEntry(g_auau_ppg_stat, "PHENIX Au+Au 200 GeV 2025", "p")

for item in ref_graphs:
    name = item["group"]["name"]
    if name in added:
        continue
    leg.AddEntry(item["stat"], name, "p")
    added.add(name)

#leg.AddEntry(g_mb_syst, "Cu+Au MB syst.", "f")

leg.Draw()

# Optional label like the plot
leg0 = root.TLegend(0.18, 0.75, 0.58, 0.96, f"     Direct #gamma ({pt_label})")
leg0.SetBorderSize(0)
leg0.SetFillStyle(0)
leg0.SetTextSize(0.05)
leg0.AddEntry(g_mb_stat, "#gamma*_{dir} 0%-93% Au+Au 200 GeV", "p")

f_scale = root.TF1("f_scale", "[0]*pow(x,[1])", 10.0, 1500.0)
f_scale.SetParameters(1e-4, 1.25)
f_scale.SetLineColor(root.kBlack)
f_scale.SetLineStyle(2)
f_scale.SetLineWidth(2)
f_scale_copy = root.TF1("f_scale_copy", "[0]*pow(x,[1])", 10.0, 1500.0)
f_scale_copy.SetParameters(1e-4, 1.25)
f_scale_copy.SetLineColor(root.kBlack)
f_scale_copy.SetLineStyle(6)
f_scale_copy.SetLineWidth(2)


# Fit using all archive reference points
g_all_fit = root.TGraphErrors(len(Nch_ref))
for i in range(len(Nch_ref)):
    g_all_fit.SetPoint(i, float(Nch_ref[i]), float(Yield_ref[i]))
    g_all_fit.SetPointError(i, 0.0, float(eYield_ref[i]))

g_all_fit.Fit(f_scale, "RQ0")
#f_scale.Draw("same")

g_auau_ppg_stat.Fit(f_scale_copy, "RQ0")
f_scale_copy.Draw("same")
leg0.AddEntry(f_scale_copy, f"#alpha^{{2025}}_{{Au+Au}} = {f_scale_copy.GetParameter(1):.2f}", "l")
#leg0.AddEntry(f_scale_copy, f"#alpha^{{2025}}_{{Au+Au}} = {f_scale_copy.GetParameter(1):.2f} #pm {f_scale_copy.GetParError(1):.2f}", "l")
#leg0.AddEntry(f_scale, f"   #alpha_{{All}} = {f_scale.GetParameter(1):.2f} #pm {f_scale.GetParError(1):.2f}", "l")
leg0.Draw()


leg2 = root.TLegend(0.6, 0.5, 0.8, 0.56,"PRC109 044912")
leg2.SetBorderSize(0)
leg2.SetFillStyle(0)
leg2.SetTextSize(0.040)
leg2.Draw()

if draw_logo: root.PHENIXTools.DrawPreliminary(0.7, 0.57, 0.23) 

c_dndy_vs_dNch.Draw()

if safe_to_pdf: c_dndy_vs_dNch.SaveAs("output/res/dndy_vs_dNch_logo.pdf")